# ADK 2.x Graph Workflows: Evaluator-Optimizer Loop

This notebook demonstrates how to build an iterative refinement loop using the ADK 2.x

**The Workflow:**
1. Takes a broad topic from the user.
2. Generates a headline.
3. Evaluates if the headline is "tech-related".
4. If unrelated, provides feedback and loops back to the generator to try again.
5. If related, the workflow successfully finishes.

### Configuration
Before initializing any ADK components, we configure the environment

In [ ]:
import os

LOCATION = "us-central1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"  # Use Agent Platform API

In [ ]:
%%bash
echo > adk_agents/.env "GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION
GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI
"

### Imports & Output Schema
We define a structured Pydantic model (`Feedback`). The evaluator agent will use this schema to guarantee its output always contains a specific `grade` and string `feedback`.

In [ ]:
from typing import Literal
from google.adk import Agent
from google.adk import Event
from google.adk import Workflow
from pydantic import BaseModel
from pydantic import Field

MODEL = "gemini-2.5-flash"

class Feedback(BaseModel):
    grade: Literal["tech-related", "unrelated"] = Field(
        description=(
            "Decide if the headline is related to technology or software"
            " engineering."
        ),
    )
    feedback: str = Field(
        description=(
            "If the headline is unrelated to technology, provide feedback on how"
            " to make it more tech-focused."
        ),
    )

### Node: Input Processor
This simple Python function takes the user's initial input and saves it to the workflow's shared memory (`state`). This allows downstream nodes to access the `{topic}` variable at any time.

In [ ]:
def process_input(node_input: str):
    """Puts user input in the state."""
    return Event(state={"topic": node_input})

### Node: Generator Agent
This LLM Agent is responsible for writing the headline. Notice the instruction template:
- `{topic}` pulls directly from the state we saved in `process_input`.
- `{feedback?}` conditionally pulls the feedback from the state. The `?` means it won't fail on the first run when feedback hasn't been generated yet.

In [ ]:
generate_headline = Agent(
    name="generate_headline",
    model=MODEL,
    instruction="""
    Write a headline about the topic "{topic}".
    If feedback is provided, take it into account.
    The feedback: {feedback?}
    """,
)

### Node: Evaluator Agent
This LLM Agent acts as the judge. It reads the previous node's output, grades it using the `Feedback` schema, and saves that Pydantic object back into the workflow state under the key `feedback`.

In [ ]:
evaluate_headline = Agent(
    name="evaluate_headline",
    model=MODEL,
    instruction="""
    Grade whether the headline is related to technology or software engineering.
    """,
    output_schema=Feedback,
    output_key="feedback",
)

### Node: Routing Logic
This Python node takes the `Feedback` object produced by the evaluator and dictates the next step in the graph by emitting an `Event(route=...)`.

In [ ]:
def route_headline(node_input: Feedback):
    return Event(route=node_input.grade)

### Assemble the Workflow
We tie the nodes together. 
- The first edge creates a linear sequence: `START -> process -> generate -> evaluate -> route`.
- The second edge creates the conditional loop: If the router emits `"unrelated"`, the graph automatically points back to `generate_headline`. 
- *Note: If it emits `"tech-related"`, there is no explicit path defined, so the graph naturally completes successfully!*

In [ ]:
root_agent = Workflow(
    name="root_agent",
    edges=[
        (
            "START",
            process_input,
            generate_headline,
            evaluate_headline,
            route_headline,
        ),
        # The Refinement Loop:
        (route_headline, {"unrelated": generate_headline}),
    ],
)

### Execute the Workflow
We run the workflow asynchronously using the `Runner` and an `InMemorySessionService`. We'll intentionally pass a topic that is clearly *not* tech-related ("baking a cake") to see the loop in action.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

# 1. Initialize session
session_service = InMemorySessionService()
session = await session_service.create_session(
    app_name="headline_app", 
    user_id="local_user",
    session_id="test_session_01"
)

runner = Runner(
    agent=root_agent,
    app_name="headline_app",
    session_service=session_service
)

Intentionally trigger a non-tech topic to force a refinement loop

In [ ]:
test_topic = "Baking a delicious chocolate cake"
input_message = Content(role="user", parts=[Part(text=test_topic)])

print(f"Initial Topic: {test_topic}\n")
print("Starting generation loop...\n")
print("-" * 50)

#Execute and stream node outputs
async for event in runner.run_async(
    user_id=session.user_id,
    session_id=session.id,
    new_message=input_message
):
    if event.message:
        print(f"[{event.node_name}] > {event.message}\n")
    elif event.content:
        # Print text generated by LLM nodes
        print(f"[{event.node_name}] > {event.content.parts[0].text}\n")

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.